# CoALA + Tool Use Design Pattern
## Customer Support Agent

---

### What This Notebook Focuses On

**Fixed (CoALA structure):** Parse -> Parallel Retrieval -> Working Memory -> Decision -> Act -> Learn

**Variable (Tool Use pattern):** Working Memory uses `create_tool_calling_agent`.
The LLM **selects** tools and their arguments. Code **executes** them.
The separation between selection and execution is the core concept.

### The key question Tool Use answers:
> *"Which tool should I call and with what arguments, given what I know?"*

### What makes Tool Use different from ReAct:
- **ReAct (NB7):** LLM must format `Thought / Action / Action Input` text — parsing is fragile
- **Tool Use (NB8):** LLM emits a structured function-call JSON — no text parsing needed
- Both are adaptive (observe and re-select), but Tool Use is more reliable and production-ready

### Memory: Pinecone `coala-memory` (seeded in Notebook_5, written to by NB6 and NB7)
Semantic and episodic memory retrieved from Pinecone inform tool selection at every step.

## CoALA Control Flow -- Tool Use Variant

```
USER MESSAGE
    |
    v
[PARSE]                  -- extract intent, order_id, sentiment
    |
    v
[PARALLEL RETRIEVAL]     -- Pinecone semantic + episodic
    |
    v
[TOOL SCHEMAS]           -- what the LLM sees (name + description + args)
    |
    v
[WORKING MEMORY]         -- LLM selects tool + args  (LLM SELECTED)
    |                       Code executes the tool     (CODE EXECUTED)
    |                       LLM sees result, selects next tool
    |                       ... repeat until Final Answer
    v
[LEARNING PHASE]         -- write facts + episode back to Pinecone
```

**Contrast with Planning (NB6):** Plan is generated once upfront, no adaptation.
**Contrast with ReAct (NB7):** ReAct parses free-form `Thought/Action` text. Tool Use uses structured JSON function calls -- more robust.

**Crucial tracing in this notebook:**
- `[LLM SELECTED]` -- printed when the LLM emits a tool call
- `[CODE EXECUTED]` -- printed when the tool runs against order.csv and returns a real result

These two tags make the selection/execution separation visible.

In [ ]:
import os, re, json, time
import pandas as pd
from typing import Optional
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")
orders_df = pd.read_csv("order.csv")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("coala-memory")
SEMANTIC_NS = "coala_semantic"
EPISODIC_NS = "coala_episodic"

print(f"Loaded {len(orders_df)} orders | Connected to Pinecone coala-memory")
print(orders_df.head(3))

In [ ]:
@tool
def fetch_order(order_id: int) -> str:
    """Fetch full order details from the database given an order ID."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    return json.dumps(result.iloc[0].to_dict(), default=str)

@tool
def check_shipping_status(order_id: int) -> str:
    """Get a human-readable shipping status message for an order."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    status = result.iloc[0]["status"]
    return {"Delivered": "Your order has been delivered.",
            "Shipped": "Your order is currently on the way.",
            "Processing": "Your order is still being prepared and has not shipped yet.",
            "Cancelled": "Your order has been cancelled."}.get(status, "Status unknown.")

@tool
def offer_compensation(order_id: int) -> str:
    """Apply a 10% discount to the customer's account as compensation for an order issue."""
    result = orders_df[orders_df["order_id"] == order_id]
    name = result.iloc[0]["user_name"] if not result.empty else "the customer"
    return f"10% discount successfully applied to {name}'s account."

@tool
def provide_order_info(order_id: int) -> str:
    """Provide detailed order information: product name, status, and order date."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    r = result.iloc[0]
    return f"Order #{r['order_id']} -- {r['product_name']}, Status: {r['status']}, Placed on: {r['date']}."

@tool
def escalate_to_human(order_id: Optional[int] = None) -> str:
    """Escalate a complex or unresolved issue to the human support team. order_id is optional."""
    suffix = f" for order {order_id}" if order_id else ""
    return f"The issue{suffix} has been escalated. A representative will contact you within 24 hours."

action_tools = [fetch_order, check_shipping_status, offer_compensation, provide_order_info, escalate_to_human]

In [ ]:
def retrieve_semantic(query: str, k: int = 3) -> list:
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = index.query(vector=qv, top_k=k, include_metadata=True, namespace=SEMANTIC_NS)
    hits = [m["metadata"]["text"] for m in res.get("matches", []) if m["score"] > 0.25]
    print(f"  [SemanticMemory] {len(hits)} facts retrieved")
    return hits

def retrieve_episodic(query: str, k: int = 2) -> list:
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = index.query(vector=qv, top_k=k, include_metadata=True, namespace=EPISODIC_NS)
    hits = [m["metadata"] for m in res.get("matches", []) if m["score"] > 0.25]
    print(f"  [EpisodicMemory] {len(hits)} past episodes retrieved")
    return hits

def learn_semantic(fact: str):
    vec = embedder.encode([fact], normalize_embeddings=True)[0].tolist()
    vid = f"fact_{int(time.time())}_{abs(hash(fact)) % 100000}"
    index.upsert(vectors=[(vid, vec, {"text": fact, "ts": int(time.time())})], namespace=SEMANTIC_NS)
    print(f"  [SemanticMemory.learn] {fact[:70]}")

def learn_episodic(episode: dict):
    summary = episode.get("summary", str(episode))
    vec = embedder.encode([summary], normalize_embeddings=True)[0].tolist()
    vid = f"ep_{int(time.time())}_{abs(hash(summary)) % 100000}"
    meta = {"summary": summary, "intent": episode.get("intent", ""),
            "outcome": episode.get("outcome", ""), "ts": int(time.time())}
    index.upsert(vectors=[(vid, vec, meta)], namespace=EPISODIC_NS)
    print(f"  [EpisodicMemory.learn] {summary[:70]}")

def parse_observation(message: str) -> dict:
    msg = message.lower()
    match = re.search(r'\b(50\d{2})\b', msg)
    order_id = int(match.group(1)) if match else None
    if any(w in msg for w in ["delay", "late", "not arrived", "not received", "not shipped"]):
        intent = "order_delay"
    elif any(w in msg for w in ["cancel", "cancellation", "cancelled"]):
        intent = "order_cancellation"
    elif any(w in msg for w in ["where", "status", "track", "when", "update", "details"]):
        intent = "order_status"
    else:
        intent = "general_enquiry"
    sentiment = "negative" if any(w in msg for w in ["unhappy", "angry", "frustrated", "unacceptable", "furious"]) else "neutral"
    print(f"  [PARSE] intent={intent}, order_id={order_id}, sentiment={sentiment}")
    return {"raw": message, "intent": intent, "order_id": order_id, "sentiment": sentiment}

## Tool Schemas -- What the LLM Sees

In the Tool Use pattern, the LLM never runs tools directly.
It receives a **schema** describing each tool's name, purpose, and required arguments.
Based on the schema and the customer message, it emits a structured **function call** (JSON).
The code then executes that call against real data.

The `print_tool_schemas()` function below makes this visible so students can see exactly
what information the LLM has when it decides which tool to call.

In [ ]:
def print_tool_schemas(tools: list):
    print("[TOOL SCHEMAS -- what the LLM sees when selecting a tool]")
    print("-" * 60)
    for t in tools:
        print(f"  Tool     : {t.name}")
        print(f"  Purpose  : {t.description}")
        print(f"  Args     : {t.args}")
        print()
    print("-" * 60)

## ToolUseTracer -- Making Selection vs Execution Visible

LangChain supports **callbacks** that fire at specific events.
We use two callbacks to print exactly when the LLM selects a tool and when the code runs it:

- `on_tool_start` fires when the LLM has emitted a function call -- the LLM selected a tool
- `on_tool_end` fires when the Python function has returned -- the code executed it

This makes the selection/execution boundary visually unmistakable in the output.

In [ ]:
class ToolUseTracer(BaseCallbackHandler):
    """Callback that prints [LLM SELECTED] and [CODE EXECUTED] for every tool call."""

    def on_tool_start(self, serialized: dict, input_str: str, **kwargs):
        tool_name = serialized.get("name", "unknown")
        print(f"\n  [LLM SELECTED]  tool='{tool_name}'  args={input_str}")
        print(f"  (LLM emitted a structured function call -- no free-text parsing needed)")

    def on_tool_end(self, output: str, **kwargs):
        preview = str(output)[:120]
        print(f"  [CODE EXECUTED] real result from order.csv = {preview}")
        print(f"  (this observation is fed back to the LLM for next selection)")

## Building the Tool Use Agent

`create_tool_calling_agent` binds the LLM to the tool schemas using the model's native
function-calling API (no prompt engineering needed for tool selection).
The `ToolUseTracer` callback is injected so every selection and execution is printed.

In [ ]:
def build_tool_use_agent(semantic_ctx: list, episodic_ctx: list) -> AgentExecutor:
    """Build a tool-calling AgentExecutor with Pinecone memory context in the system prompt."""
    semantic_block = "\n".join(f"- {f}" for f in semantic_ctx) if semantic_ctx else "None retrieved"
    episodic_block = "\n".join(e.get("summary", "") for e in episodic_ctx) if episodic_ctx else "None retrieved"

    system_prompt = f"""You are a customer support agent using structured tool calls.

Long-term memory retrieved from Pinecone:
Semantic facts:
{semantic_block}

Past similar episodes:
{episodic_block}

Use the available tools to resolve the customer's issue.
Always fetch the order first when an order ID is present.
After seeing the order status, choose the most appropriate follow-up action.
Give an empathetic final answer once you have enough information."""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    agent = create_tool_calling_agent(llm=llm, tools=action_tools, prompt=prompt)
    return AgentExecutor(
        agent=agent,
        tools=action_tools,
        verbose=False,               # ToolUseTracer handles our custom printing
        handle_parsing_errors=True,
        max_iterations=6,
        callbacks=[ToolUseTracer()],
    )

## CoALA Tool Use Agent

In [ ]:
class CoALAToolUseAgent:

    def handle(self, message: str) -> str:
        print(f"\nUSER: {message}")
        print("=" * 60)

        # 1. PARSE
        print("\n[PARSE]")
        parsed = parse_observation(message)

        # 2. PARALLEL RETRIEVAL from Pinecone
        print("\n[PARALLEL RETRIEVAL -- Pinecone coala-memory]")
        semantic_ctx = retrieve_semantic(message)
        episodic_ctx = retrieve_episodic(message)

        # 3. SHOW TOOL SCHEMAS -- what the LLM sees before selecting
        print("\n[TOOL SCHEMAS -- LLM uses these to decide what to call]")
        print_tool_schemas(action_tools)

        # 4. WORKING MEMORY -- tool selection + execution loop
        print("[WORKING MEMORY -- Tool Use loop]")
        print("  [LLM SELECTED] printed when LLM emits a function call")
        print("  [CODE EXECUTED] printed when Python runs the tool against order.csv")
        agent_executor = build_tool_use_agent(semantic_ctx, episodic_ctx)
        result = agent_executor.invoke({"input": message})
        response = result["output"]

        # 5. LEARNING PHASE
        print("\n[LEARNING PHASE]")
        learn_semantic(f"Tool Use resolved {parsed['intent']} via structured function calls")
        learn_episodic({
            "summary": f"{message[:60]} -> Tool Use resolution, intent={parsed['intent']}",
            "intent": parsed["intent"],
            "outcome": "resolved",
        })

        print(f"\nFINAL RESPONSE:\n{response}")
        return response

## Run 1 -- Order ID Present

Customer provides order 5001. Agent should:
1. `[LLM SELECTED]` fetch_order(5001) -- LLM sees order_id in message, picks correct tool
2. `[CODE EXECUTED]` real CSV row returned
3. `[LLM SELECTED]` follow-up tool based on status observed
4. Final answer grounded in real data

Notice how every `[LLM SELECTED]` line is a **structured JSON call**, not free-text.
The LLM never guesses the result -- it always waits for `[CODE EXECUTED]`.

In [ ]:
agent = CoALAToolUseAgent()

# Order 5001 exists in CSV -- agent fetches real data and responds based on actual status
agent.handle("I want to know the status of my order 5001 and whether I can get any compensation.")

## Run 2 -- No Order ID (Agent Must Escalate)

Customer does not provide an order ID. No order lookup is possible.
Watch the agent select `escalate_to_human` without an order_id -- it cannot fetch data it doesn't have.

**Key lesson:** The agent is grounded. It does not fabricate an order ID or invent a result.
When it lacks the information to call a data tool, it uses a safe fallback (`escalate_to_human`).
This is the **grounding property** of Tool Use -- hallucination is prevented by requiring real tool results.

In [ ]:
# No order ID in message -- agent cannot call fetch_order, must escalate
agent.handle("My package was supposed to arrive last week and it still hasn't shown up. I am very frustrated.")

## Tool Use vs ReAct -- Side-by-Side

Both NB7 and NB8 are adaptive (the agent can change course based on tool results).
The difference is HOW the tool is selected:

| Aspect | ReAct (NB7) | Tool Use (NB8) |
|---|---|---|
| How LLM selects a tool | Writes `Action: tool_name` as free text | Emits structured JSON function call |
| How tool is identified | Text parsing (`Action:` line extracted) | Parsed directly from JSON schema |
| Risk of wrong tool call | Parsing can fail on malformed text | Near-zero -- structured format |
| What LLM sees | Full prompt with `{tools}` block | Bound function schemas via API |
| Adaptability | Yes -- Observation informs next Thought | Yes -- tool result informs next selection |

**In production, Tool Use (function calling) is preferred over ReAct for reliability.**
ReAct is useful when you need visible reasoning traces or work with models that don't support function calling.

## Key Observations -- Tool Use Pattern

**What to notice in the output:**
- `[TOOL SCHEMAS]` block shows exactly what the LLM receives before making any decision
- `[LLM SELECTED]` fires the moment the LLM emits a function call -- structured JSON, no parsing
- `[CODE EXECUTED]` fires when Python runs the tool against real order.csv data
- When no order ID is present, the agent selects `escalate_to_human` -- it doesn't fabricate data
- Each `[CODE EXECUTED]` result is fed back to the LLM, grounding the next selection

**Cost vs benefit:**
- More LLM calls than Planning (NB6), similar to ReAct (NB7)
- Far more reliable than ReAct in production because function calls are structured
- Hallucination is prevented -- the LLM must wait for real tool results

---

## Full Comparison: Planning vs ReAct vs Tool Use

| Aspect | Planning (NB6) | ReAct (NB7) | Tool Use (NB8) |
|---|---|---|---|
| LLM calls per interaction | 1 (upfront plan) | N (one per Thought step) | N (one per tool selection) |
| Can adapt mid-execution | No -- plan is fixed | Yes -- Observation drives next Thought | Yes -- tool result drives next selection |
| Tool selection method | JSON plan generated upfront | Free-text `Action:` line parsed | Structured function call via API |
| Execution by | Code (plan runner) | LangChain ReAct executor | LangChain tool-calling executor |
| Risk of wrong action | Plan may be wrong from start | Text parsing can fail | Near-zero -- structured JSON |
| Grounding | Yes (tools run against CSV) | Yes (tools run against CSV) | Yes (tools run against CSV) |
| Production readiness | Medium (rigid) | Medium (fragile parsing) | High (structured + adaptive) |
| Best for | Known, stable workflows | Uncertain tasks, visible reasoning | Any grounded real-data task |